In [ ]:
import pandas as pd
import numpy as np
import keras
from keras import models
from keras import layers
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 1000
(X_train, y_train), (X_test, y_test) = reuters.load_data(num_words=vocab_size)

print("Data loaded successfully")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Data loaded successfully
Training samples: 8982
Testing samples: 2246


In [ ]:
# help(imdb.get_word_index)

In [ ]:
# Word index mapping (word -> integer index)
word_index = reuters.get_word_index()

# Reverse mapping (integer index -> word), needed to decode reviews
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(encoded_review):
    # indices are offset by 3 because 0,1,2 are reserved for padding, start-of-seq, unknown
    return " ".join([reverse_word_index.get(i-3, "?") for i in encoded_review])

In [ ]:
# for (key, value) in word_index.items():
#   print(key, value)
#   break

In [ ]:
# reverse_word_index[1], reverse_word_index[14], reverse_word_index[22], reverse_word_index[16], reverse_word_index[43]

In [ ]:
for i in range(5):
    print(f"Review {i+1}")
    print("Label:", "Positive" if y_train[i] == 1 else "Negative")
    print("Encoded review:\n", X_train[i])
    print("\nDecoded review:\n", decode_review(X_train[i]))
    print("-" * 100)

Review 1
Label: Negative
Encoded review:
 [1, 2, 2, 8, 43, 10, 447, 5, 25, 207, 270, 5, 2, 111, 16, 369, 186, 90, 67, 7, 89, 5, 19, 102, 6, 19, 124, 15, 90, 67, 84, 22, 482, 26, 7, 48, 4, 49, 8, 864, 39, 209, 154, 6, 151, 6, 83, 11, 15, 22, 155, 11, 15, 7, 48, 9, 2, 2, 504, 6, 258, 6, 272, 11, 15, 22, 134, 44, 11, 15, 16, 8, 197, 2, 90, 67, 52, 29, 209, 30, 32, 132, 6, 109, 15, 17, 12]

Decoded review:
 ? ? ? said as a result of its december acquisition of ? co it expects earnings per share in 1987 of 1 15 to 1 30 dlrs per share up from 70 cts in 1986 the company said pretax net should rise to nine to 10 mln dlrs from six mln dlrs in 1986 and ? ? revenues to 19 to 22 mln dlrs from 12 5 mln dlrs it said cash ? per share this year should be 2 50 to three dlrs reuter 3
----------------------------------------------------------------------------------------------------
Review 2
Label: Negative
Encoded review:
 [1, 2, 699, 2, 2, 56, 2, 2, 9, 56, 2, 2, 81, 5, 2, 57, 366, 737, 132, 20, 2, 7, 

In [ ]:
review_lengths = [len(review) for review in X_train]

shortest_len = min(review_lengths)
longest_len = max(review_lengths)
mean_len = int(np.mean(review_lengths))

print("Length of shortest review:", shortest_len)
print("Length of longest review:", longest_len)
print("Average (mean) review length:", mean_len)

Length of shortest review: 13
Length of longest review: 2376
Average (mean) review length: 145


In [ ]:
maxlen = 100

X_train_padded = pad_sequences(X_train, maxlen=maxlen, padding='post', value=0)
X_test_padded = pad_sequences(X_test, maxlen=maxlen, padding='post', value=0)

print("Padding length used (mean review length):", maxlen)
print("Shape of padded training data:", X_train_padded.shape)
print("Shape of padded testing data:", X_test_padded.shape)
print("\nExample padded review (review 1):\n", X_train_padded[0])

Padding length used (mean review length): 100
Shape of padded training data: (8982, 100)
Shape of padded testing data: (2246, 100)

Example padded review (review 1):
 [  1   2   2   8  43  10 447   5  25 207 270   5   2 111  16 369 186  90
  67   7  89   5  19 102   6  19 124  15  90  67  84  22 482  26   7  48
   4  49   8 864  39 209 154   6 151   6  83  11  15  22 155  11  15   7
  48   9   2   2 504   6 258   6 272  11  15  22 134  44  11  15  16   8
 197   2  90  67  52  29 209  30  32 132   6 109  15  17  12   0   0   0
   0   0   0   0   0   0   0   0   0   0]


In [ ]:
print("Number of training records:", len(X_train))
print("Number of testing records:", len(X_test))

train_pos = np.sum(y_train == 1)
train_neg = np.sum(y_train == 0)
test_pos = np.sum(y_test == 1)
test_neg = np.sum(y_test == 0)

print("\nTraining set -> Positive:", train_pos, " Negative:", train_neg)
print("Testing set  -> Positive:", test_pos, " Negative:", test_neg)

Number of training records: 8982
Number of testing records: 2246

Training set -> Positive: 432  Negative: 55
Testing set  -> Positive: 105  Negative: 12


In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, GlobalAveragePooling1D, Dense
from keras.optimizers import Adam

model = Sequential([
    Embedding(input_dim=1000, output_dim=30, input_length=X_train_padded.shape[1]),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(46, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.fit(
    X_train_padded,
    y_train,
    epochs=100,
    batch_size=16,
    verbose=0
)

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_padded,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Test Loss: 1.55393385887146
Test Accuracy: 0.7528940439224243
